# Silver Layer — Products
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.products`, applies cleaning and business logic,
and writes the curated result to `salesflow_dev.silver.products`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Remove duplicates by `ProductID` |
| 2 | Clean `ProductName` |
| 3 | Fill nulls in numeric columns with `0` |
| 4 | Add `is_available` flag |
| 5 | Add `price_category` classification |
| 6 | Add `data_quality_status` flag |
| 7 | Add `processing_timestamp` |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import current_timestamp, when, col

# Read products table from Bronze layer
df = spark.table("salesflow_dev.bronze.products")

print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Remove Duplicates
Deduplicate by primary key `ProductID`.

In [0]:
# Drop duplicate ProductIDs — primary key must be unique in Silver
df = df.dropDuplicates(["ProductID"])

print(f"Records after deduplication: {df.count()}")

## 3. Clean and Standardize Columns
Applies string cleaning and fills nulls in numeric columns.

In [0]:
# 3.1 Clean ProductName using shared utility
df = clean_string_column(df, "ProductName")

# 3.2 Fill nulls in numeric columns with 0
df = df.fillna({
    "UnitsInStock": 0,
    "UnitsOnOrder": 0,
    "ReorderLevel": 0
})

## 4. Add Business Logic Columns

### `is_available`
`TRUE` if `UnitsInStock > 0`, otherwise `FALSE`.

### `price_category`
| Range | Category |
|---|---|
| `UnitPrice < 10` | Low |
| `10 ≤ UnitPrice ≤ 50` | Medium |
| `UnitPrice > 50` | High |

In [0]:
# 4.1 is_available: TRUE if product has stock
df = df.withColumn(
    "is_available",
    when(col("UnitsInStock") > 0, True).otherwise(False)
)

# 4.2 price_category: classify products by unit price
df = df.withColumn(
    "price_category",
    when(col("UnitPrice") < 10, "Low")
    .when((col("UnitPrice") >= 10) & (col("UnitPrice") <= 50), "Medium")
    .when(col("UnitPrice") > 50, "High")
    .otherwise(None)  # null UnitPrice gets no category
)

## 5. Add Quality Flag
Marks records as `INVALID` if `ProductName` is null **or** `UnitPrice` is negative.  
Otherwise marks as `VALID`.

In [0]:
# Custom quality condition: invalid if ProductName is null OR UnitPrice is negative
# Note: add_quality_flag() only checks for nulls, so we handle UnitPrice separately
df = df.withColumn(
    "data_quality_status",
    when(
        col("ProductName").isNull() | (col("UnitPrice") < 0),
        "INVALID"
    ).otherwise("VALID")
)

## 6. Add Processing Timestamp

In [0]:
# Capture when this record was processed in the Silver layer
df = df.withColumn("processing_timestamp", current_timestamp())

## 7. Save as Delta Table

In [0]:
# Write to Silver layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.silver.products")

print("Table saved: salesflow_dev.silver.products")

## 8. Validation

In [0]:
silver_products = spark.table("salesflow_dev.silver.products")

# Record count
print(f"Total records: {silver_products.count()}")

# Quality flag distribution
print("\nQuality flag distribution:")
display(silver_products.groupBy("data_quality_status").count())

# Availability distribution
print("\nAvailability distribution:")
display(silver_products.groupBy("is_available").count())

# Price category distribution
print("\nPrice category distribution:")
display(silver_products.groupBy("price_category").count().orderBy("price_category"))

# Schema
print("\nSchema:")
silver_products.printSchema()

# Sample
print("\nFirst 5 rows:")
display(silver_products.limit(5))